# 03 - Eksperimen: Skenario B, Skenario C, dan uji kebocoran

- **B**: fitur URL-only (18 fitur leksikal dari string URL), random split 80:20.
- **C**: fitur URL-only yang sama, tetapi split per kelompok domain (`GroupShuffleSplit`).
- **A tanpa USI**: seperti A, tetapi `URLSimilarityIndex` dikeluarkan (uji kebocoran).
- **Ulangan seed**: B dan C diulang dengan beberapa seed untuk melihat variasi hasil.

Notebook 02 harus dijalankan lebih dulu (Skenario A tersimpan di `results/metrics.csv`).

In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent          # folder repo (notebook ada di folder notebooks/)
sys.path.insert(0, str(ROOT))

from src.preprocessing import (SEED, Y_COL, GROUP_COL, load_data, feature_sets,
                               split_random, split_group)
from src.evaluation import run_scenario, save_results

DATA_PATH = ROOT / "data" / "PhiUSIIL_Phishing_URL_Dataset.csv"
RESULTS_PATH = ROOT / "results" / "metrics.csv"
FIG_DIR = ROOT / "figures"
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

## 1. Muat data

In [2]:
df, info = load_data(DATA_PATH)
print(info)
# Cek arti label: label 1 = legitimate, label 0 = phishing. y_phishing: 1 = phishing.
print(pd.crosstab(df["label"], df[Y_COL], rownames=["label (Kaggle)"], colnames=["y_phishing"]))

{'baris_awal': 235795, 'url_duplikat_dibuang': 425, 'baris_akhir': 235370}
y_phishing           0       1
label (Kaggle)                
0                    0  100520
1               134850       0


## 2. Fitur URL-only (B dan C)
Hanya fitur yang dihitung dari string URL. Empat fitur turunan statistik (`URLSimilarityIndex`, `CharContinuationRate`, `TLDLegitimateProb`, `URLCharProb`) dan semua fitur dari isi halaman HTML tidak dipakai.

In [3]:
fs = feature_sets(df)
print(len(fs["B"]), "fitur URL-only")
print(fs["B"])
# Fitur turunan statistik: URLSimilarityIndex bernilai 100 untuk semua baris legitimate
print(df.groupby("label")["URLSimilarityIndex"].agg(["min", "mean", "max", "std"]))

18 fitur URL-only
['URLLength', 'DomainLength', 'IsDomainIP', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS']
              min        mean    max        std
label                                          
0        0.155574   49.651466  100.0  22.578566
1      100.000000  100.000000  100.0   0.000000


## 3. Skenario B (URL-only, random split)

In [4]:
tr, te = split_random(df, test_size=0.2, seed=SEED)
res_B = run_scenario(df, "B", fs["B"], tr, te, split_name="random 80:20 (stratified)",
                     seed=SEED, grouped_cv=False, do_cv=True, fig_dir=FIG_DIR,
                     notes="URL-only (18 fitur); URL duplikat dibuang")
res_B.drop(columns=["scenario", "split", "features", "notes"]).round(5)

,model,n_features,seed,val_f1,val_roc_auc,test_f1,test_precision,test_recall,test_roc_auc,inference_ms_per_sample,n_train,n_test
0,LogisticRegression,18,42,0.99520,0.99828,0.99565,0.99970,0.99164,0.99841,0.00038,188296,47074
1,RandomForest,18,42,0.99678,0.99833,0.99711,0.99905,0.99518,0.99843,0.00739,188296,47074


## 4. Skenario C (URL-only, split per domain)
Satu domain hanya boleh ada di train **atau** test. Cross-validation di dalam train juga memakai `GroupKFold` per domain.

In [5]:
tr_c, te_c = split_group(df, test_size=0.2, seed=SEED)
overlap = set(df.loc[tr_c, GROUP_COL]) & set(df.loc[te_c, GROUP_COL])
print("train:", len(tr_c), "| test:", len(te_c), "| domain yang muncul di keduanya:", len(overlap))
print("proporsi phishing di train:", round(df.loc[tr_c, Y_COL].mean(), 4),
      "| di test:", round(df.loc[te_c, Y_COL].mean(), 4))
res_C = run_scenario(df, "C", fs["C"], tr_c, te_c, split_name="GroupShuffleSplit per domain 80:20",
                     seed=SEED, grouped_cv=True, do_cv=True, fig_dir=FIG_DIR,
                     notes="URL-only (18 fitur); grup = domain tanpa www.; URL duplikat dibuang")
res_C.drop(columns=["scenario", "split", "features", "notes"]).round(5)

train: 188086 | test: 47284 | domain yang muncul di keduanya: 0
proporsi phishing di train: 0.4275 | di test: 0.4255


,model,n_features,seed,val_f1,val_roc_auc,test_f1,test_precision,test_recall,test_roc_auc,inference_ms_per_sample,n_train,n_test
0,LogisticRegression,18,42,0.99537,0.99835,0.99488,0.9995,0.99031,0.9981,0.00026,188086,47284
1,RandomForest,18,42,0.99693,0.99831,0.99661,0.9991,0.99413,0.9982,0.01175,188086,47284


## 5. Uji kebocoran: Skenario A tanpa `URLSimilarityIndex`

In [6]:
tr, te = split_random(df, test_size=0.2, seed=SEED)
res_A2 = run_scenario(df, "A_tanpa_USI", fs["A_tanpa_USI"], tr, te,
                      split_name="random 80:20 (stratified)", seed=SEED,
                      grouped_cv=False, do_cv=True, fig_dir=FIG_DIR,
                      notes="semua fitur numerik kecuali URLSimilarityIndex")
res_A2.drop(columns=["scenario", "split", "features", "notes"]).round(5)

,model,n_features,seed,val_f1,val_roc_auc,test_f1,test_precision,test_recall,test_roc_auc,inference_ms_per_sample,n_train,n_test
0,LogisticRegression,49,42,0.99910,0.99999,0.99913,0.99965,0.99861,0.99996,0.00063,188296,47074
1,RandomForest,49,42,0.99985,0.99999,0.99975,0.99980,0.99970,1.00000,0.00849,188296,47074


## 6. Simpan hasil utama (seed 42)

In [7]:
save_results(pd.concat([res_B, res_C, res_A2], ignore_index=True), RESULTS_PATH).round(5)

,scenario,model,split,features,n_features,seed,val_f1,val_roc_auc,test_f1,test_precision,test_recall,test_roc_auc,inference_ms_per_sample,n_train,n_test,notes
0,A,LogisticRegression,random 80:20 (stratified),semua,50,42,0.99986,1.00000,0.99980,1.00000,0.99960,0.99999,0.00064,188296,47074,semua fitur numerik; URL duplikat dibuang
1,A,RandomForest,random 80:20 (stratified),semua,50,42,0.99999,1.00000,1.00000,1.00000,1.00000,1.00000,0.00599,188296,47074,semua fitur numerik; URL duplikat dibuang
2,B,LogisticRegression,random 80:20 (stratified),URL-only,18,42,0.99520,0.99828,0.99565,0.99970,0.99164,0.99841,0.00038,188296,47074,URL-only (18 fitur); URL duplikat dibuang
3,B,RandomForest,random 80:20 (stratified),URL-only,18,42,0.99678,0.99833,0.99711,0.99905,0.99518,0.99843,0.00739,188296,47074,URL-only (18 fitur); URL duplikat dibuang
4,C,LogisticRegression,GroupShuffleSplit per domain 80:20,URL-only,18,42,0.99537,0.99835,0.99488,0.99950,0.99031,0.99810,0.00026,188086,47284,URL-only (18 fitur); grup = domain tanpa www.;...
5,C,RandomForest,GroupShuffleSplit per domain 80:20,URL-only,18,42,0.99693,0.99831,0.99661,0.99910,0.99413,0.99820,0.01175,188086,47284,URL-only (18 fitur); grup = domain tanpa www.;...
6,A_tanpa_USI,LogisticRegression,random 80:20 (stratified),semua,49,42,0.99910,0.99999,0.99913,0.99965,0.99861,0.99996,0.00063,188296,47074,semua fitur numerik kecuali URLSimilarityIndex
7,A_tanpa_USI,RandomForest,random 80:20 (stratified),semua,49,42,0.99985,0.99999,0.99975,0.99980,0.99970,1.00000,0.00849,188296,47074,semua fitur numerik kecuali URLSimilarityIndex


## 7. Ulangan dengan beberapa seed (B dan C, tanpa cross-validation)
Untuk melihat apakah selisih antara B dan C stabil. Hasilnya disimpan di `results/metrics_seeds.csv`.

In [8]:
SEEDS = [42, 1, 2, 3, 4]
rows = []
for s in SEEDS:
    tr, te = split_random(df, 0.2, s)
    rows.append(run_scenario(df, "B", fs["B"], tr, te, "random 80:20 (stratified)", seed=s, do_cv=False))
    tr, te = split_group(df, 0.2, s)
    rows.append(run_scenario(df, "C", fs["C"], tr, te, "GroupShuffleSplit per domain 80:20", seed=s, do_cv=False))
seeds_df = pd.concat(rows, ignore_index=True)
seeds_df.to_csv(ROOT / "results" / "metrics_seeds.csv", index=False)

ringkas = (seeds_df.groupby(["scenario", "model"])[["test_f1", "test_roc_auc", "test_recall"]]
           .agg(["mean", "std"]).round(4))
ringkas

test_f1         test_roc_auc         test_recall        
                               mean     std         mean     std        mean     std
scenario model                                                                      
B        LogisticRegression  0.9954  0.0003       0.9983  0.0002      0.9913  0.0006
         RandomForest        0.9970  0.0002       0.9983  0.0001      0.9949  0.0003
C        LogisticRegression  0.9951  0.0003       0.9981  0.0002      0.9906  0.0007
         RandomForest        0.9968  0.0002       0.9982  0.0001      0.9944  0.0003

## Catatan untuk dijelaskan saat verifikasi
1. Kenapa 4 fitur turunan statistik tidak dimasukkan ke fitur URL-only?
2. Kenapa `www.` dibuang dari kolom Domain sebelum dikelompokkan?
3. Kenapa Skenario C memakai `GroupKFold` untuk cross-validation di dalam train?
4. Apa arti hasil kalau selisih B dan C kecil? (sebagian besar domain hanya punya satu URL)
5. Waktu inferensi di sini mengukur apa, dan apa yang tidak diukur?